## RAG Explainability

In this notebook, a Retreival Augmented Generation pipeline is implemented to fetch similar articles from sample to explain the classification reasoning of our model in natural language.

### Importing and loading artifacts

We start with importing our best model metadata and our samples.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline

# Load config and artifacts
with open("models/best_config.json") as f:
    cfg = json.load(f)

with open("models/best_model_meta.json") as f:
    meta = json.load(f)

vectorizer = joblib.load("models/best_vectorizer.pkl")
model      = joblib.load(meta["file"])

# Load data
train_df = pd.read_csv("Dataset/train.csv")
test_df  = pd.read_csv("Dataset/test.csv")

train_df[cfg["input_field"]] = train_df[cfg["input_field"]].fillna("")
test_df[cfg["input_field"]]  = test_df[cfg["input_field"]].fillna("")

print("Model type :", meta["model_type"])
print("Best F1    :", meta["best_f1"])
print("Train size :", len(train_df))
print("Test size  :", len(test_df))


Model type : logistic_regression
Best F1    : 0.9937
Train size : 33043
Test size  : 11015


### Embed training set and Build FIASS Index

The training data is then vectorized using TF-IDF vectorizer selected.

Our main backbone for RAG is to fetch similar content from training set. However a brute-force cosine similarity search for each test sample is slow. We use Facebook AI Similarity Search (FAISS), opensource, for optimized rettrieval from our vectorized training set.

In [2]:
print("Vectorizing training set...")
X_train_tfidf = vectorizer.transform(train_df[cfg["input_field"]])

# Convert to dense float32 — FAISS requires dense arrays
X_train_dense = X_train_tfidf.toarray().astype(np.float32)


Vectorizing training set...


Building FAISS flat index with the same size as our 50,000 columns. Each index represents the words(phrases) and FAISS uses flat indexing to refer to them later on.

In [3]:
# Normalize so cosine similarity = dot product (faster FAISS search)
faiss.normalize_L2(X_train_dense)

# Build FAISS flat index (exact nearest neighbor search)
dim   = X_train_dense.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product = cosine on L2-normalized vectors
index.add(X_train_dense)

print(f"FAISS index built.")
print(f"  Vectors indexed : {index.ntotal}")
print(f"  Vector dimension: {dim}")

FAISS index built.
  Vectors indexed : 33043
  Vector dimension: 50000


### TF-IDF key phrase extractor

We then define a function that extracts feature words(phrases) from their TF-IDF score. This will be used when performing similarity check and retreiving exact words mentioned that indicated the validity of the news article.

In [4]:
def get_top_tfidf_phrases(text, vectorizer, top_n=10):
    vec = vectorizer.transform([text])
    indices = np.argsort(vec.data)[::-1][:top_n]
    feature_names = np.array(vectorizer.get_feature_names_out())
    top_features = feature_names[vec.indices[indices]]
    top_scores   = vec.data[indices]
    return list(zip(top_features, top_scores.round(4)))

sample_text = test_df[cfg["input_field"]].iloc[0]
phrases = get_top_tfidf_phrases(sample_text, vectorizer)
print("Top TF-IDF phrases:")
for phrase, score in phrases:
    print(f"  {phrase:<30} {score}")


Top TF-IDF phrases:
  clark                          0.4447
  gun                            0.2861
  minneapolis                    0.2595
  march                          0.1825
  got gun                        0.1634
  paramedic                      0.1549
  lee                            0.1355
  black                          0.1337
  girlfriend                     0.124
  flag                           0.1163


### Human readable explainer model

To explain the prediction of the model using natural language, a pretrained model `google/flan-t5-large` was used from `HuggingFace` to download and run the model locally.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer  = AutoTokenizer.from_pretrained("google/flan-t5-large")
llm_model  = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

def generate_explanation(prompt, max_new_tokens=200):
    inputs  = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_explanation("Is the sky blue? Answer yes or no and explain briefly."))


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


The sky is blue because it is a color of light. Light is blue because it is reflected off of the surface of the Earth. The answer: yes.


### RAG Pipeline

Next, the RAG pipeline is defined with the procedures:
- Vectorize the test news article
- Retrieve top-k similar training samples and check their class label
- Extract the phrases having top tfidf score on the test article because their tfidf score was used to get the similar training samples and hence what our model uses to decide
- Construct natural language prompt for explanation directions

In [6]:
def rag_explain(text, label_pred, top_k=3):
    # 1. Vectorize and L2-normalize query
    vec = vectorizer.transform([text]).toarray().astype(np.float32)
    faiss.normalize_L2(vec)

    # 2. Retrieve top-k similar training examples
    D, I = index.search(vec, top_k)
    retrieved = train_df.iloc[I[0]]

    # 3. Top TF-IDF phrases
    phrases    = get_top_tfidf_phrases(text, vectorizer, top_n=5)
    phrase_str = ", ".join([f"'{p}'" for p, _ in phrases])

    # 4. Neighbor label stats
    neighbor_labels  = retrieved["label"].tolist()
    same_label_count = sum(1 for l in neighbor_labels if l == label_pred)
    label_str        = "FAKE" if label_pred == 0 else "REAL"

    # 5. Model confidence
    confidence = model.predict_proba(vectorizer.transform([text]))[0][label_pred] * 100

    # 6. Template explanation (deterministic, always complete)
    template = (
        f"This article is classified as {label_str} with {confidence:.1f}% confidence. "
        f"The most distinctive phrases found were: {phrase_str}. "
        f"These phrases appear frequently in {label_str} articles in the training data. "
        f"{same_label_count} out of {top_k} most similar training articles were also "
        f"labeled {label_str}, further supporting this prediction."
    )

    # 7. LLM adds a short tone/topic description (small, constrained task)
    snippet     = text[:150].replace("\n", " ")
    llm_prompt  = f"In one short sentence, describe the topic of this news snippet: \"{snippet}\""
    llm_tone    = generate_explanation(llm_prompt, max_new_tokens=60)
    explanation = template + f" Topic summary: {llm_tone}"

    return explanation, phrases, retrieved[["label", cfg["input_field"]]].head(top_k)

# Smoke test
sample_idx  = 1
sample_text = test_df[cfg["input_field"]].iloc[sample_idx]
sample_pred = model.predict(vectorizer.transform([sample_text]))[0]
true_label  = test_df["label"].iloc[sample_idx]

explanation, phrases, neighbors = rag_explain(sample_text, sample_pred)
print(f"True label : {'REAL' if true_label == 1 else 'FAKE'}")
print(f"Prediction : {'REAL' if sample_pred == 1 else 'FAKE'}")
print(f"\nExplanation:\n{explanation}")


True label : REAL
Prediction : REAL

Explanation:
This article is classified as REAL with 99.9% confidence. The most distinctive phrases found were: 'muslimmajority country', 'muslimmajority', 'trump', 'refugee people', 'order'. These phrases appear frequently in REAL articles in the training data. 3 out of 3 most similar training articles were also labeled REAL, further supporting this prediction. Topic summary: trump fights back amid inte


### Demo

Below is the demonstration of 5 test samples and output of our RAG pipeline.

In [7]:
sample_indices = [10, 20, 30, 40, 50]  
fake_indices = test_df[test_df["label"] == 0].index[:3].tolist()
real_indices = test_df[test_df["label"] == 1].index[:2].tolist()
sample_indices = fake_indices + real_indices

for idx in sample_indices:
    text       = test_df.loc[idx, cfg["input_field"]]
    true_label = test_df.loc[idx, "label"]
    pred       = model.predict(vectorizer.transform([text]))[0]

    explanation, phrases, neighbors = rag_explain(text, pred)

    true_str = "REAL" if true_label == 1 else "FAKE"
    pred_str = "REAL" if pred == 1 else "FAKE"
    match    = "✓" if true_label == pred else "✗ WRONG"

    print("=" * 70)
    print(f"True: {true_str}  |  Predicted: {pred_str}  {match}")
    print(f"Article snippet: {text[:100].replace(chr(10), ' ')}...")
    print(f"\n{explanation}")
    print()


True: FAKE  |  Predicted: FAKE  ✓
Article snippet: black felon brutally beat girlfriend grab cop gun stick gun cop backcops partner shoot kill feloncop...

This article is classified as FAKE with 99.9% confidence. The most distinctive phrases found were: 'clark', 'gun', 'minneapolis', 'march', 'got gun'. These phrases appear frequently in FAKE articles in the training data. 3 out of 3 most similar training articles were also labeled FAKE, further supporting this prediction. Topic summary: mnbur : black felon brutally beat girlfriend grab cop gun stick gun cop backcops partner shoot kill felon

True: FAKE  |  Predicted: FAKE  ✓
Article snippet: anything provoke trump block twitter even ice cream guess trump missed stick stone lesson life criti...

This article is classified as FAKE with 98.8% confidence. The most distinctive phrases found were: 'trump', 'blocked', 'twitter', 'ben jerry', 'flavor'. These phrases appear frequently in FAKE articles in the training data. 2 out of 3 most sim